# Ajuste de bosque aleatorio y validación cruzada

A lo largo de los siguientes ejercicios, aprenderás a usar Python para construir y validar un modelo de conjunto de bosque aleatorio con scikit-learn. Antes de comenzar con este ejercicio de programación, recomendamos encarecidamente ver la conferencia en video y completar la IVQ para los temas asociados.

**Nota**: Este cuaderno está dividido en dos partes, cada una con su propio video instructivo. Comienza aquí, o pasa directamente a [Parte 2](#part_2) si ya has completado la primera parte.


Todo la información que necesitas para resolver esta tarea está en este cuaderno, y todo el código que implementarás tendrá lugar dentro de este cuaderno.


Temas de enfoque incluyen:


*   Declaraciones de importación relevantes
*   Codificación de características categóricas como variables ficticias
*   Estratificación durante la división de datos
*   Ajuste de un modelo
*   Uso de `GridSearchCV` para validar el modelo mediante validación cruzada y ajustar los siguientes hiperparámetros:  
    - `max_depth`  
    - `max_features`  
    - `min_samples_split`
    - `n_estimators`  
    - `min_samples_leaf`  
*   Evaluación del modelo usando precisión, recall y puntuación f1


## Revisión

Este cuaderno es una continuación del proyecto de rotación bancaria. A continuación se presenta un resumen de las consideraciones y decisiones que ya hemos tomado. Para una discusión detallada de estos temas, consulte el [cuaderno anotado completo para el modelo de árbol de decisión](https://www.coursera.org/learn/the-nuts-and-bolts-of-machine-learning/ungradedLab/HVyMU/annotated-follow-along-guide-build-a-decision-tree).

>  **Objetivo del modelado:** Predecir si un cliente abandonará&mdash;una tarea de clasificación binaria.

>  **Variable objetivo:** columna `Exited&mdash;0 o 1`.

>  **Equilibrio de clases:** Los datos están desequilibrados 80/20 (no abandonó/abandonó), pero no realizaremos balanceo de clases.

>  **Métrica de evaluación principal:** Puntuación F1.

>  **Flujo de trabajo de modelado y selección de modelo:** El modelo campeón será el que tenga la mejor puntuación F1 de validación. Solo se usará el modelo campeón para predecir en los datos de prueba. Consulte el cuaderno anotado del árbol de decisión para detalles y limitaciones de este enfoque.


## Una nota sobre validación cruzada/validación

Este cuaderno es para fines educativos. Como tal, incluye dos enfoques para la validación: validación cruzada de los datos de entrenamiento y validación usando un conjunto de validación separado. En la práctica, generalmente solo usarás uno u otro para un proyecto dado.

La validación cruzada es más rigurosa, porque maximiza el uso de los datos de entrenamiento, pero si tienes un conjunto de datos muy grande o recursos informáticos limitados, puede ser mejor validar con un conjunto de validación separado.


## Declaraciones de importación

Antes de comenzar con los ejercicios y analizar los datos, necesitamos importar todas las bibliotecas y extensiones requeridas para este ejercicio de programación. A lo largo del curso, utilizaremos numpy y pandas para operaciones, y matplotlib para graficar.


In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# This lets us see all of the columns, preventing Juptyer from redacting them.
pd.set_option('display.max_columns', None)

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score,\
f1_score, confusion_matrix, ConfusionMatrixDisplay

from sklearn.ensemble import RandomForestClassifier

# This module lets us save our models once we fit them.
import pickle

## Leer los datos


In [2]:
# Read in data
file = 'Churn_Modelling.csv'
df_original = pd.read_csv(file)
df_original.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Ingeniería de características


### Selección de características

En este paso, prepararemos los datos para el modelado.  Observe que desde arriba hay varias columnas que no esperaríamos que ofrezcan ninguna señal predictiva al modelo. Estas columnas incluyen `RowNumber`, `CustomerID`, y `Surname`. Eliminaremos estas columnas para que no introduzcan ruido a nuestro modelo.  

También eliminaremos la columna `Gender`, porque no queremos que nuestro modelo haga predicciones basadas en el género.


In [3]:
# Drop useless and sensitive (Gender) cols
churn_df = df_original.drop(['RowNumber', 'CustomerId', 'Surname', 'Gender'], axis=1)
churn_df.head()

,CreditScore,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,41,1,83807.86,1,0,1,112542.58,0
2,502,France,42,8,159660.80,3,1,0,113931.57,1
3,699,France,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,43,2,125510.82,1,1,1,79084.10,0


### Transformación de características

A continuación, codificaremos en variables ficticias la variable `Geography`, que es categórica. Hacemos esto con la función `pd.get_dummies()` y estableciendo `drop_first='True'`, lo que reemplaza la columna `Geography` con dos nuevas columnas Boolean llamadas `Geography_Germany` y `Geography_Spain`.


In [4]:
# Dummy encode categoricals
churn_df2 = pd.get_dummies(churn_df, drop_first='True')
churn_df2.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,619,42,2,0.00,1,1,1,101348.88,1,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1


## Dividir los datos

Dividiremos los datos en características y variable objetivo, y en datos de entrenamiento y datos de prueba usando la función `train_test_split()`.

No olvides incluir el parámetro `stratify=y`, ya que esto asegura que la proporción de clases 80/20 de la variable objetivo se mantenga en ambos conjuntos de datos de entrenamiento y prueba después de la división.

Por último, establecemos una semilla aleatoria para que nosotros y otros podamos reproducir nuestro trabajo.


In [5]:
# Define the y (target) variable
y = churn_df2["Exited"]

# Define the X (predictor) variables
X = churn_df2.copy()
X = X.drop("Exited", axis = 1)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

## Modelado


### Ajuste de hiperparámetros con validación cruzada

El proceso de validación cruzada es el mismo que fue para el modelo de árbol de decisión. La única diferencia es que ahora estamos ajustando más hiperparámetros. Los pasos se incluyen a continuación como una revisión.

Para detalles sobre la validación cruzada con `GridSearchCV`, consulte de nuevo el cuaderno del árbol de decisión, o la [documentación de GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV) en scikit-learn.

1. Instanciar el clasificador (y establecer el `random_state`).

2. Crear un diccionario de hiperparámetros para buscar.

3. Crear un conjunto de métricas de puntuación para capturar.

4. Instanciar el objeto `GridSearchCV`. Pasar como argumentos:
  - El clasificador (`rf`)
  - El diccionario de hiperparámetros para buscar (`cv_params`)
  - El conjunto de métricas de puntuación (`scoring`)
  - El número de pliegues de validación cruzada que deseas (`cv=5`)
  - La métrica de puntuación que quieres que GridSearch utilice cuando seleccione el modelo "mejor" (es decir, el modelo que funciona mejor en promedio en todos los pliegues de validación) (`refit='f1'`)

5. Ajustar los datos (`X_train`, `y_train`) al objeto `GridSearchCV` (`rf_cv`).

Tenga en cuenta que usamos la magia `%%time` en la parte superior de la celda. Esto muestra el tiempo de ejecución final de la celda. (Los comandos mágicos, a menudo llamados simplemente "magics," son comandos que están integrados en IPython para simplificar tareas comunes. Comienzan con `%` o `%%`.)


In [6]:
%%time

rf = RandomForestClassifier(random_state=0)

cv_params = {'max_depth': [2,3,4,5, None], 
             'min_samples_leaf': [1,2,3],
             'min_samples_split': [2,3,4],
             'max_features': [2,3,4],
             'n_estimators': [75, 100, 125, 150]
             }  

scoring = {'accuracy', 'precision', 'recall', 'f1'}

rf_cv = GridSearchCV(rf, cv_params, scoring=scoring, cv=5, refit='f1')

# rf_cv.fit(X_train, y_train)

CPU times: user 44 µs, sys: 22 µs, total: 66 µs
Wall time: 70.3 µs


This is the end of the first video. The next video will continue from this point.


<a name="part_2"></a>
# Validación de bosque aleatorio en conjunto de datos separado

Esta es una continuación del cuaderno de modelado de bosque aleatorio validado cruzado para la rotación de clientes bancarios. Esta sección del cuaderno demostrará cómo construir y validar un modelo de conjunto de bosque aleatorio en Python con scikit-learn.


Los temas de enfoque incluyen:

  * Usar `pickle` para guardar un modelo ajustado
  * Usar un conjunto de datos separado para ajustar hiperparámetros y validar tu modelo
    * Dividir los datos de entrenamiento para crear un conjunto de validación
    * Crear una lista de índices de división para usar con `PredefinedSplit` para que `GridSearchCV` realice validación en este conjunto de validación definido


## Pickle  

Cuando los modelos tardan mucho en ajustarse, no quieres tener que ajustarlos más de una vez. Si tu kernel se desconecta o apagas el cuaderno y pierdes la salida de la celda, tendrás que volver a ajustar el modelo, lo cual puede ser frustrante y llevar mucho tiempo. 

`pickle` es una herramienta que guarda el objeto del modelo ajustado en una ubicación especificada, y luego lo lee rápidamente. También te permite usar modelos que fueron ajustados en otro lugar, sin tener que entrenarlos tú mismo.


In [7]:
# Define a path to the folder where you want to save the model
path = '/home/jovyan/work/'

Este paso va a ***Escribir*** (es decir, guardar) el modelo, en ***Binario*** (por lo tanto, `wb`), en la carpeta designada por la ruta anterior. En este caso, el nombre del archivo que estamos escribiendo es `rf_cv_model.pickle`.


In [8]:
# Pickle the model
with open(path+'rf_cv_model.pickle', 'wb') as to_write:
    pickle.dump(rf_cv, to_write)

Una vez que guardemos el modelo, nunca tendremos que volver a ajustarlo cuando ejecutemos este cuaderno. Idealmente, podríamos abrir el cuaderno, seleccionar "Ejecutar todo", y las celdas se ejecutarían con éxito hasta el final sin volver a entrenar el modelo.

Para que esto suceda, necesitaremos volver a la celda donde definimos nuestra búsqueda en cuadrícula y comentar la línea donde ajustamos el modelo. De lo contrario, cuando volvamos a ejecutar el cuaderno, volvería a ajustar el modelo.

De manera similar, también necesitaremos volver a donde guardamos el modelo como un pickle y comentar esas líneas.

A continuación, agregaremos una nueva celda que lea el modelo guardado desde la carpeta que ya especificamos. Para esto, usaremos `rb` (leer en modo binario) y aseguraremos de asignar el modelo al mismo nombre de variable que usamos arriba, `rf_cv`.


In [9]:
# Read in pickled model
with open(path + 'rf_cv_model.pickle', 'rb') as to_read:
    rf_cv = pickle.load(to_read)

Ahora todo lo anterior está listo para ejecutarse rápidamente y sin volver a ajustar. Podemos continuar usando el atributo `best_params_` del modelo para verificar los hiperparámetros que tuvieron la mejor puntuación F1 promedio en todas las particiones de validación cruzada.


In [10]:
rf_cv.fit(X_train, y_train)
rf_cv.best_params_

{'max_depth': None,
 'max_features': 4,
 'min_samples_leaf': 2,
 'min_samples_split': 2,
 'n_estimators': 125}

Y para verificar la mejor puntuación media de F1 de este modelo en los pliegues de validación, podemos usar el atributo `best_score_`. Recuerda, si en su lugar hubiéramos establecido `refit=recall` cuando instanciamos nuestro objeto `GridSearchCV` anteriormente, entonces llamar a `best_score_` devolvería la mejor puntuación de recall, y los mejores parámetros podrían no ser los mismos que los que están en la celda anterior, porque el modelo estaría optimizando para una métrica diferente.


In [11]:
rf_cv.best_score_

0.580528563620339

Nuestro modelo tuvo una puntuación F1 de 0.5805&mdash;no es terrible. Recuerda que cuando ejecutamos nuestra búsqueda en cuadrícula, especificamos que también queríamos capturar precisión, recall y exactitud.

La razón para hacer esto es que es difícil interpretar una puntuación F1. Estas otras métricas son mucho más directamente interpretables, por lo que vale la pena conocerlas.

La siguiente celda define una función auxiliar que extrae estas puntuaciones del objeto `GridSearchCV` ajustado y devuelve un marco de datos de pandas con las cuatro puntuaciones del modelo con la mejor puntuación F1 promedio durante la validación.


In [12]:
def make_results(model_name, model_object):
    '''
    Accepts as arguments a model name (your choice - string) and
    a fit GridSearchCV model object.

    Returns a pandas df with the F1, recall, precision, and accuracy scores
    for the model with the best mean F1 score across all validation folds.
    '''

    # Get all the results from the CV and put them in a df
    cv_results = pd.DataFrame(model_object.cv_results_)

    # Isolate the row of the df with the max(mean f1 score)
    best_estimator_results = cv_results.iloc[cv_results['mean_test_f1'].idxmax(), :]

    # Extract accuracy, precision, recall, and f1 score from that row
    f1 = best_estimator_results.mean_test_f1
    recall = best_estimator_results.mean_test_recall
    precision = best_estimator_results.mean_test_precision
    accuracy = best_estimator_results.mean_test_accuracy

    # Create table of results
    table = pd.DataFrame({'Model': [model_name],
                          'F1': [f1],
                          'Recall': [recall],
                          'Precision': [precision],
                          'Accuracy': [accuracy]
                         }
                        )

    return table

In [13]:
# Make a results table for the rf_cv model using above function
rf_cv_results = make_results('Random Forest CV', rf_cv)
rf_cv_results

,Model,F1,Recall,Precision,Accuracy
0,Random Forest CV,0.580529,0.472517,0.756289,0.861333


Podemos concatenar estos resultados a nuestra tabla maestra de resultados desde cuando construimos el modelo de árbol de decisión único.


In [14]:
# Read in master results table
results = pd.read_csv('results1.csv', index_col=0)
results

,Model,F1,Recall,Precision,Accuracy
0,Tuned Decision Tree,0.560655,0.469255,0.701608,0.8504


In [15]:
# Concatenate the random forest results to the master table
results = pd.concat([rf_cv_results, results])
results

,Model,F1,Recall,Precision,Accuracy
0,Random Forest CV,0.580529,0.472517,0.756289,0.861333
0,Tuned Decision Tree,0.560655,0.469255,0.701608,0.850400


Las puntuaciones en la tabla anterior nos indican que el modelo de bosque aleatorio funciona mejor que el modelo de árbol de decisión único en cada métrica. ¡Genial!

Ahora, construyamos otro modelo de bosque aleatorio, solo que esta vez ajustaremos los hiperparámetros usando un conjunto de validación separado.


## Modelado


### Hiperparámetros ajustados con conjunto de validación separado

Comienza dividiendo los datos de entrenamiento para crear un conjunto de validación. Recuerda, no vamos a tocar los datos de prueba en absoluto.

Usaremos `train_test_split` para dividir `X_train` y `y_train` en 80% de datos de entrenamiento (`X_tr`, `y_tr`) y 20% de datos de validación (`X_val`, `y_val`). No olvides estratificarlo y establecer el estado aleatorio.


In [16]:
# Create separate validation data
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, 
                                            stratify=y_train, random_state=10)

Cuando ajustamos hiperparámetros con `GridSearchCV` usando un conjunto de validación separado, tenemos que tomar algunos pasos adicionales. `GridSearchCV` quiere validar los datos cruzadamente. De hecho, si el argumento `cv` se dejara en blanco, dividiría los datos en cinco pliegues para validación cruzada por defecto.

No queremos que haga esto. En cambio, vamos a decirle exactamente qué filas de `X_train` son para entrenamiento, y qué filas son para validación.

Para hacer esto, necesitamos hacer una lista de longitud `len(X_train)` donde cada elemento sea 0 o -1. Un 0 en el índice _i_ indicará a `GridSearchCV` que el índice _i_ de `X_train` debe mantenerse fuera para validación. Un -1 en un índice dado indicará que ese índice de `X_train` debe usarse como datos de entrenamiento.

Haremos esta lista usando una comprensión de listas que mira el número de índice de cada fila en `X_train`. Si ese número de índice está en la lista de números de índice de `X_val`, entonces la comprensión de listas añade un 0. Si no, entonces añade un -1.

Entonces, si nuestros datos de entrenamiento son:  
[A, B, C, D],  
y nuestra lista es:  
[-1, 0, 0, -1],  
entonces `GridSearchCV` usará un conjunto de entrenamiento de [A, D] y un conjunto de validación de [B, C].


In [17]:
# Create list of split indices
split_index = [0 if x in X_val.index else -1 for x in X_train.index]

Ahora que tenemos esta lista, necesitamos importar una nueva función llamada `PredefinedSplit`. Esta función es lo que nos permite pasar la lista que acabamos de hacer a `GridSearchCV`. (Puedes leer más sobre esta función en la [documentación](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.PredefinedSplit.html#sklearn.model_selection.PredefinedSplit).)


In [18]:
from sklearn.model_selection import PredefinedSplit

Ahora podemos construir el modelo. Todo es igual que cuando validamos cruzadamente, excepto que esta vez pasamos la lista `split_index` a la función `PredefinedSplit` y la asignamos a una nueva variable llamada `custom_split`.

Luego usaremos esta variable para el argumento `cv` cuando instanciamos `GridSearchCV`.


In [19]:
rf = RandomForestClassifier(random_state=0)

cv_params = {'max_depth': [2,3,4,5, None], 
             'min_samples_leaf': [1,2,3],
             'min_samples_split': [2,3,4],
             'max_features': [2,3,4],
             'n_estimators': [75, 100, 125, 150]
             }  

scoring = {'accuracy', 'precision', 'recall', 'f1'}

custom_split = PredefinedSplit(split_index)

rf_val = GridSearchCV(rf, cv_params, scoring=scoring, cv=custom_split, refit='f1')

Ahora ajusta el modelo.


In [20]:
rf_val.fit(X_train, y_train)

CPU times: user 3min 51s, sys: 1.2 s, total: 3min 53s
Wall time: 3min 53s


GridSearchCV(cv=PredefinedSplit(test_fold=array([-1,  0, ..., -1, -1])),
             error_score=nan,
             estimator=RandomForestClassifier(bootstrap=True, ccp_alpha=0.0,
                                              class_weight=None,
                                              criterion='gini', max_depth=None,
                                              max_features='auto',
                                              max_leaf_nodes=None,
                                              max_samples=None,
                                              min_impurity_decrease=0.0,
                                              min_impurity_split=None,
                                              min_samples_leaf=1,
                                              min_samples_split=2,
                                              min_weigh...
                                              oob_score=False, random_state=0,
                                              verbose=0, warm_

Observe que esto tomó menos tiempo que cuando validamos cruzadamente&mdash;alrededor de 1/5 del tiempo. Esto se debe a que _durante la validación cruzada_ los datos de entrenamiento se dividieron en cinco pliegues. Se cultivó un conjunto de árboles con una combinación particular de hiperparámetros en cuatro pliegues de datos, y se validó en el quinto pliegue que se reservó. Todo este proceso ocurrió para cada uno de los cinco pliegues de reserva. Luego, se entrenó otro conjunto con la siguiente combinación de hiperparámetros, repitiendo todo el proceso. Esto continuó hasta que no hubo más combinaciones de hiperparámetros para ejecutar.  

<img src="./cross_validation_diagram.svg"/>

Pero ahora que estamos _usando un conjunto de validación separado,_ se construye un conjunto para cada combinación de hiperparámetros. Cada conjunto se entrena en el nuevo conjunto de entrenamiento y se valida en el conjunto de validación. Pero esto solo sucede _una vez_ para cada combinación de hiperparámetros, en lugar de _cinco veces_ con validación cruzada. Por eso, el tiempo de entrenamiento fue solo una quinta parte más largo.

<img src="./single_validation_diagram.svg"/>

Vamos a guardar el modelo...


In [21]:
# Pickle the model
with open(path+'rf_val_model.pickle', 'wb') as to_write:
    pickle.dump(rf_val, to_write)

... y comentar dónde ajustamos el modelo y escribimos el pickle, luego leímos de nuevo el modelo enlatado.


In [22]:
# Open pickled model
with open(path+'rf_val_model.pickle', 'rb') as to_read:
    rf_val = pickle.load(to_read)

Ahora verifica los parámetros del modelo con mejor rendimiento en el conjunto de validación:


In [23]:
rf_val.best_params_

{'max_depth': None,
 'max_features': 4,
 'min_samples_leaf': 1,
 'min_samples_split': 3,
 'n_estimators': 150}

Notice that the best hyperparameters were slightly different than the cross-validated model.  

Now, we'll generate the model results using the `make_results` function, add them to the master table, and then sort them by F1 score in descending order.


In [24]:
# Create model results table
rf_val_results = make_results('Random Forest Validated', rf_val)

# Concatentate model results table with master results table
results = pd.concat([rf_val_results, results])

# Sort master results by F1 score in descending order
results.sort_values(by=['F1'], ascending=False)

,Model,F1,Recall,Precision,Accuracy
0,Random Forest CV,0.580529,0.472517,0.756289,0.861333
0,Random Forest Validated,0.575510,0.460784,0.766304,0.861333
0,Tuned Decision Tree,0.560655,0.469255,0.701608,0.850400


Podemos guardar la nueva tabla maestra para usarla más tarde cuando construyamos más modelos.


In [25]:
# Save the master results table
results.to_csv(path+'results2.csv', index=False);

## Selección de modelos y resultados finales

Ahora tenemos tres modelos. Si hemos decidido que hemos terminado de intentar optimizarlos, entonces ahora podemos usar nuestro mejor modelo para predecir en los datos de reserva de prueba. Usaremos el modelo validado cruzadamente sin la limitación de profundidad, pero si en cambio usáramos el modelo que fue validado contra un conjunto de validación separado, ahora volveríamos y volveríamos a entrenar el modelo en el conjunto completo de entrenamiento (conjuntos de entrenamiento + validación).

**Nota**: _Podría ser tentador ver cómo todos los modelos funcionan en los datos de reserva de prueba, y luego elegir el que mejor funciona. Aunque esto **puede** hacerse, sesga el modelo final, porque usaste tus datos de prueba para volver y tomar una decisión anterior. Los datos de prueba deben representar datos **no vistos**. En competencias, por ejemplo, debes enviar tu modelo final antes de recibir los datos de prueba._

Los resultados en la tabla anterior nos dicen que el modelo de bosque aleatorio validado cruzadamente funciona un poco mejor que el entrenado en un conjunto de validación separado.

Funciona bien para precisión y exactitud, pero la recuperación es 0.4725. Esto significa que de todas las personas en los pliegues de validación que _realmente_ dejaron el banco, el modelo identifica con éxito el 47% de ellas.

Aún no aplicaremos el modelo a los datos de prueba, porque todavía hay un modelo más que construir. Pronto aprenderás sobre esto. Una vez que entrenemos ese modelo, usaremos nuestro modelo campeón para predecir en los datos de prueba.


**¡Felicidades!** Has completado este laboratorio. Sin embargo, es posible que no notes una marca de verificación verde junto a este elemento en la plataforma de Coursera. Por favor, continúa tu progreso independientemente de la marca de verificación. Solo haz clic en el icono de "guardar" en la parte superior de este cuaderno para asegurarte de que tu trabajo ha sido registrado.

Ahora entiendes cómo usar Python para construir y validar un modelo de conjunto de bosques aleatorios. En adelante, puedes comenzar a usar Python para construir y validar modelos de conjunto de bosques aleatorios con tus propios datos.
